# TASK 4 · Sentiment Analysis

Build a machine learning system to classify tweets as positive, negative, or neutral using NLP preprocessing, TF-IDF, Naive Bayes, and Logistic Regression.

In [ ]:
import re, string, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from wordcloud import WordCloud
sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

In [ ]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

In [ ]:
DATA_PATH = "Tweets.csv"
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
print(df.columns.tolist())

## Inspect sentiment distribution

The target column is `airline_sentiment`, containing positive, neutral, and negative classes.

In [ ]:
print(df.isnull().sum().sort_values(ascending=False).head(10))
print(df["airline_sentiment"].value_counts())

In [ ]:
counts=df["airline_sentiment"].value_counts()
plt.figure(figsize=(8,5)); sns.barplot(x=counts.index,y=counts.values); plt.title("Sentiment Distribution"); plt.xlabel("Sentiment"); plt.ylabel("Number of Tweets"); plt.show()

**Observation:** This chart shows the distribution of positive, neutral, and negative tweets. Class imbalance should be considered when interpreting accuracy.

## Text preprocessing

The pipeline lowercases text, removes URLs, mentions, punctuation and numbers, tokenizes, removes stopwords, and lemmatizes words.

In [ ]:
stop_words=set(stopwords.words("english")); lemmatizer=WordNetLemmatizer()
def preprocess_text(text):
    text=str(text).lower()
    text=re.sub(r"http\S+|www\S+", " ", text)
    text=re.sub(r"@\w+", " ", text)
    text=re.sub(r"#", "", text)
    text=text.translate(str.maketrans("", "", string.punctuation))
    tokens=re.findall(r"\b[a-zA-Z]+\b", text)
    return " ".join(lemmatizer.lemmatize(t) for t in tokens if t not in stop_words)
df=df.dropna(subset=["text","airline_sentiment"]).copy()
df["clean_text"]=df["text"].apply(preprocess_text)
display(df[["text","clean_text","airline_sentiment"]].head(10))

**Observation:** Cleaning reduces textual noise while preserving important sentiment-bearing words.

## TF-IDF Feature Extraction

**TF-IDF (Term Frequency–Inverse Document Frequency)** converts text into numerical features. TF measures how often a term appears in a document, while IDF downweights terms that occur across many documents. This helps models emphasize words that distinguish sentiment classes.

In [ ]:
X=df["clean_text"]; y=df["airline_sentiment"]
X_train_text,X_test_text,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
tfidf=TfidfVectorizer(max_features=15000,ngram_range=(1,2),min_df=2,sublinear_tf=True)
X_train=tfidf.fit_transform(X_train_text); X_test=tfidf.transform(X_test_text)
print("Train:",X_train.shape,"Test:",X_test.shape)

## Model 1 — Multinomial Naive Bayes

In [ ]:
nb_model=MultinomialNB(); nb_model.fit(X_train,y_train); nb_pred=nb_model.predict(X_test)
print(classification_report(y_test,nb_pred))

## Model 2 — Logistic Regression

In [ ]:
lr_model=LogisticRegression(max_iter=1000,random_state=42); lr_model.fit(X_train,y_train); lr_pred=lr_model.predict(X_test)
print(classification_report(y_test,lr_pred))

In [ ]:
def metrics(ytrue,ypred,name):
    return [name,accuracy_score(ytrue,ypred),precision_score(ytrue,ypred,average="weighted",zero_division=0),recall_score(ytrue,ypred,average="weighted",zero_division=0),f1_score(ytrue,ypred,average="weighted",zero_division=0)]
results=pd.DataFrame([metrics(y_test,nb_pred,"Naive Bayes"),metrics(y_test,lr_pred,"Logistic Regression")],columns=["Model","Accuracy","Precision","Recall","F1-Score"])
display(results.round(4))

**Observation:** Compare the two models using all four metrics. Weighted F1 is especially useful when classes are not perfectly balanced.

## Confusion Matrices

In [ ]:
labels=["negative","neutral","positive"]
for name,pred in [("Naive Bayes",nb_pred),("Logistic Regression",lr_pred)]:
    cm=confusion_matrix(y_test,pred,labels=labels)
    plt.figure(figsize=(7,5)); sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=labels,yticklabels=labels); plt.title(f"{name} Confusion Matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()

**Observation:** Diagonal cells are correct predictions; off-diagonal cells show which sentiments are confused.

## WordClouds by Sentiment

In [ ]:
for sentiment in labels:
    text=" ".join(df.loc[df.airline_sentiment==sentiment,"clean_text"].dropna())
    wc=WordCloud(width=900,height=450,background_color="white",max_words=100).generate(text)
    plt.figure(figsize=(10,5)); plt.imshow(wc,interpolation="bilinear"); plt.axis("off"); plt.title(f"WordCloud — {sentiment.capitalize()} Sentiment"); plt.show()

**Observation:** WordClouds reveal frequently occurring words associated with each sentiment class and provide qualitative context.

## Error Analysis

In [ ]:
best_name=results.loc[results["F1-Score"].idxmax(),"Model"]; best_pred=nb_pred if best_name=="Naive Bayes" else lr_pred
errors=pd.DataFrame({"Text":X_test_text.values,"Actual":y_test.values,"Predicted":best_pred})
errors=errors[errors.Actual!=errors.Predicted]
print("Best model:",best_name); display(errors.head(5))

### Why errors happen

Common causes include sarcasm, mixed sentiment, very short tweets, context-dependent airline language, and neutral wording that overlaps with positive or negative vocabulary. Review the five displayed examples and discuss the likely cause for each.

## Conclusion

This project applies NLP preprocessing, TF-IDF, Multinomial Naive Bayes, and Logistic Regression to classify tweets as positive, neutral, or negative. The preferred model is the one with the strongest overall evaluation metrics.

### Real-world applications
- Customer feedback monitoring
- Brand reputation analysis
- Social media monitoring
- Airline/service quality analysis
- Product review analysis
- Customer support prioritization

## Submission Notes

- Keep `Tweets.csv` in the same folder as the notebook.
- Use the 80/20 stratified train/test split.
- Review all metrics, confusion matrices, WordClouds, and five error examples before submission.